In [1]:
import logging
import pathlib
from pprint import pprint

from dotenv import load_dotenv

# ── Load API keys before any LLM call ─────────────────────────────────────────
_env = pathlib.Path().resolve().parent / 'src' / 'krrood' / 'llmr' / 'workflows' / '.env'
if not _env.exists():
    _env = pathlib.Path().resolve() / '.env'
load_dotenv(_env, override=True)

logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s: %(message)s')
logging.getLogger('krrood').setLevel(logging.DEBUG)

# ── krrood.llmr core ──────────────────────────────────────────────────────────
from krrood.llmr import (
    ActionPipeline,
    ActionDispatcher,
    ActionSpec,
    EntityGrounder,
    ExecutionLoop,
    TaskDecomposer,
    ClarificationNeededError,
    ActionSlotSchema,
    EntityDescriptionSchema,
)

# ── pycram bridge (optional — imports pycram + sdt internally) ─────────────────
from krrood.llmr.pycram_bridge import PycramBridge

print('All imports OK')

All imports OK


In [2]:
from llmr import load_pr2_apartment_world
from semantic_digital_twin.world_description.world_entity import Body

world, pr2, context = load_pr2_apartment_world()
print(f'World loaded. groundable_type = {Body}')

WARNING polytope.solvers: `polytope` failed to import `cvxopt.glpk`.
WARNING polytope.solvers: will use `scipy.optimize.linprog`
Unknown attribute "type" in /robot[@name='pr2']/link[@name='base_laser_link']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='wide_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='narrow_stereo_optical_frame']
Unknown attribute "type" in /robot[@name='pr2']/link[@name='laser_tilt_link']
Unknown tag "material" in /robot[@name='pr2']/link[@name='l_force_torque_link']/collision[1]
Unknown tag "material" in /robot[@name='apartment']/link[@name='coffe_machine']/collision[1]


World loaded. groundable_type = <class 'semantic_digital_twin.world_description.world_entity.Body'>


In [3]:
bridge = PycramBridge(
    groundable_type=Body,
    context={
        # Manipulators keyed by arm name (used by AutoActionHandler for Manipulator fields)
        'manipulators': {
            'LEFT':  pr2.left_arm.manipulator,
            'RIGHT': pr2.right_arm.manipulator,
        },
    },
)

print(f'Registered action types: {bridge.action_types}')

INFO krrood.llmr.pycram_bridge.action_registry: PycramActionRegistry: discovered 14 action(s): ['CloseAction', 'OpenAction', 'DetectAction', 'LookAtAction', 'NavigateAction', 'GraspingAction', 'PickUpAction', 'ReachAction', 'PlaceAction', 'CarryAction', 'FollowToolCenterPointPathAction', 'MoveTorsoAction', 'ParkArmsAction', 'SetGripperAction']
INFO krrood.llmr.pycram_bridge: PycramBridge ready — 14 action(s): ['CloseAction', 'OpenAction', 'DetectAction', 'LookAtAction', 'NavigateAction', 'GraspingAction', 'PickUpAction', 'ReachAction', 'PlaceAction', 'CarryAction', 'FollowToolCenterPointPathAction', 'MoveTorsoAction', 'ParkArmsAction', 'SetGripperAction']


Registered action types: ['CloseAction', 'OpenAction', 'DetectAction', 'LookAtAction', 'NavigateAction', 'GraspingAction', 'PickUpAction', 'ReachAction', 'PlaceAction', 'CarryAction', 'FollowToolCenterPointPathAction', 'MoveTorsoAction', 'ParkArmsAction', 'SetGripperAction']


In [4]:
bridge.registry.action_types_dict()

{'CloseAction': 'Closes a container like object.',
 'OpenAction': 'Opens a container like object',
 'DetectAction': 'Detects an object that fits the object description and returns an object designator_description describing the object.\n\nIf no object is found, an PerceptionObjectNotFound error is raised.',
 'LookAtAction': 'Lets the robot look at a position.',
 'NavigateAction': 'Navigates the Robot to a position.',
 'GraspingAction': 'Grasps an object described by the given Object Designator description',
 'PickUpAction': 'Let the robot pick up an object.',
 'ReachAction': 'Let the robot reach a specific pose.',
 'PlaceAction': 'Places an Object at a position using an arm.',
 'CarryAction': "Parks the robot's arms. And align the arm with the given Axis of a frame.",
 'FollowToolCenterPointPathAction': "Represents an action to move a robotic arm's TCP (Tool Center Point) along a\npath of poses.",
 'MoveTorsoAction': 'Move the torso of the robot up and down.',
 'ParkArmsAction': 'Park 

In [5]:
bridge.registry.schemas()['PickUpAction'].fields

[FieldSpec(name='object_designator', raw_type=<class 'semantic_digital_twin.world_description.world_entity.Body'>, kind=<FieldKind.ENTITY: 'entity'>, docstring='Object designator_description describing the object that should be picked up', is_optional=False, default=<object object at 0x71fa1db61f20>, enum_members=[], sub_fields=[]),
 FieldSpec(name='arm', raw_type=<enum 'Arms'>, kind=<FieldKind.ENUM: 'enum'>, docstring='The arm that should be used for pick up', is_optional=False, default=<object object at 0x71fa1db61f20>, enum_members=['LEFT', 'RIGHT', 'BOTH'], sub_fields=[]),
 FieldSpec(name='grasp_description', raw_type=<class 'pycram.datastructures.grasp.GraspDescription'>, kind=<FieldKind.COMPLEX: 'complex'>, docstring='The GraspDescription that should be used for picking up the object', is_optional=False, default=<object object at 0x71fa1db61f20>, enum_members=[], sub_fields=[FieldSpec(name='approach_direction', raw_type=<enum 'ApproachDirection'>, kind=<FieldKind.ENUM: 'enum'>, d

## 4. Inspect introspected action schemas

In [6]:
for action_type, schema in bridge.registry.schemas().items():
    print(f'\n=== {action_type} ===')
    print(f'  doc: {schema.docstring[:80]}...' if len(schema.docstring) > 80 else f'  doc: {schema.docstring}')
    for f in schema.fields:
        opt = '?' if f.is_optional else ' '
        members = f' [{", ".join(f.enum_members)}]' if f.enum_members else ''
        print(f'  {opt} {f.name:30s} {f.kind.value:10s}{members}')


=== CloseAction ===
  doc: Closes a container like object.
    object_designator              entity    
    arm                            enum       [LEFT, RIGHT, BOTH]
  ? grasping_prepose_distance      primitive 

=== OpenAction ===
  doc: Opens a container like object
    object_designator              entity    
    arm                            enum       [LEFT, RIGHT, BOTH]
  ? grasping_prepose_distance      primitive 

=== DetectAction ===
  doc: Detects an object that fits the object description and returns an object designa...
    technique                      enum       [ALL, HUMAN, TYPES, REGION, HUMAN_ATTRIBUTES, HUMAN_WAVING]
  ? state                          primitive 
  ? object_sem_annotation          primitive 
  ? region                         primitive 

=== LookAtAction ===
  doc: Lets the robot look at a position.
    target                         pose      
  ? camera                         entity    

=== NavigateAction ===
  doc: Navigates the Robot to 

## 5. Build ActionPipeline from bridge

`bridge.make_pipeline()` returns an `ActionPipeline` pre-configured with introspection-
driven slot-filling — the LLM is told the exact field names and types for every action.

In [7]:
pipeline = bridge.make_pipeline()

print('ActionPipeline ready')
print(f'  groundable_type : {pipeline.groundable_type.__name__}')
print(f'  action_types    : {list(pipeline.action_types.keys())}')

ActionPipeline ready
  groundable_type : Body
  action_types    : ['CloseAction', 'OpenAction', 'DetectAction', 'LookAtAction', 'NavigateAction', 'GraspingAction', 'PickUpAction', 'ReachAction', 'PlaceAction', 'CarryAction', 'FollowToolCenterPointPathAction', 'MoveTorsoAction', 'ParkArmsAction', 'SetGripperAction']


## 6. Inspect world context (what the LLM sees)

In [8]:
from krrood.llmr.pipeline.action_pipeline import serialise_world_from_symbol_graph

world_ctx = serialise_world_from_symbol_graph(Body)
print(world_ctx)

## World State Summary

Scene objects and surfaces: base_footprint, map, odom_combined, apartment_root, furnitures_root, walls, windows, wall_coloksu_wall1, wall_coloksu_wall2, wall_coloksu_wall3, wall_coloksu_wall4, wardrobe, wardrobe_door_left, wardrobe_door_left_handle, wardrobe_door_right, wardrobe_door_right_handle, armchair, sofa, coffee_table, coffee_table_drawer, bedside_table, kitchen_root, side_A, side_B, cabinet1, cabinet1_door_top_left, handle_cab1_top_door, oven, cabinet1_drawer_middle, handle_cab1_drawer_mid, cabinet1_drawer_bottom, handle_cab1_drawer_bottom, cabinet1_coloksu_level4, cabinet1_coloksu_level5, cabinet2, cabinet2_door_out_fancy, cabinet2_door_left, handle_cab2_door, cabinet2_drawer_small, cabinet2_drawer_big, cabinet3, cabinet3_door_top_out_fancy, cabinet3_door_top_left, handle_cab3_door_top, cabinet3_door_bottom_out_fancy, cabinet3_door_bottom_left, handle_cab3_door_bottom, cabinet4, cabinet4_door_top, cabinet4_door_bottom, handle_cab4_door_bottom, cabinet5

## 7. Phase 1 — classify_and_extract (slot filling)

The slot-filler now uses the introspected schemas, so it knows exactly which roles
(`object_designator`, `arm`, etc.) to extract from the instruction.

In [9]:
schema = pipeline.classify_and_extract('pick up the milk from the table')
schema.model_dump()

INFO krrood.llmr.pipeline.action_pipeline: classify_and_extract – action_type=PickUpAction, entities=['milk']


{'action_type': 'PickUpAction',
 'entities': [{'role': 'object_designator',
   'name': 'milk',
   'semantic_type': 'Milk',
   'spatial_context': 'from the table',
   'attributes': None}],
 'parameters': {},
 'manner': None,
 'constraints': None}

In [10]:
# place_schema = pipeline.classify_and_extract('place the milk on the island countertop')
# pprint(place_schema.model_dump())

## 8. Phase 2 — dispatch (grounding + parameter resolution → ActionSpec)

In [11]:
spec = pipeline.dispatch(schema)
print(f'action_type       : {spec.action_type}')
print(f'parameters        : {spec.parameters}')
print(f'grounded_entities : {list(spec.grounded_entities.keys())}')
print()
pprint(spec)

INFO krrood.llmr.pipeline.dispatcher: Dispatching to 'PickUpAction' handler.
INFO krrood.llmr.pipeline.action_pipeline: ActionPipeline.dispatch complete – PickUpAction resolved.


action_type       : PickUpAction
parameters        : {'arm': BOTH, 'grasp_description': GraspDescription(approach_direction=<ApproachDirection.FRONT: (<AxisIdentifier.X: (1, 0, 0)>, -1)>, vertical_alignment=<VerticalAlignment.TOP: (<AxisIdentifier.Z: (0, 0, 1)>, -1)>, manipulator=Body(name=PrefixedName('None/milk.stl'), id=UUID('2207bc9f-88b2-4393-b374-10c47da55db4'), index=219), rotate_gripper=False, manipulation_offset=0.05)}
grounded_entities : ['object_designator']

ActionSpec(action_type='PickUpAction',
           parameters={'arm': BOTH,
                       'grasp_description': GraspDescription(approach_direction=<ApproachDirection.FRONT: (<AxisIdentifier.X: (1, 0, 0)>, -1)>,
                                                             vertical_alignment=<VerticalAlignment.TOP: (<AxisIdentifier.Z: (0, 0, 1)>, -1)>,
                                                             manipulator=Body(name=PrefixedName('None/milk.stl'),
                                                  

## 9. Convert ActionSpec → pycram action instance

`bridge.to_pycram_action(spec)` merges resolved parameters and grounded entities and
calls the pycram action constructor — ready for `.perform()`.

In [12]:
pycram_action = bridge.to_pycram_action(spec)
print(f'pycram action type  : {type(pycram_action).__name__}')
print(f'pycram action       : {pycram_action}')

pycram action type  : PickUpAction
pycram action       : PickUpAction(object_designator=Body(name=PrefixedName('None/milk.stl'), id=UUID('2207bc9f-88b2-4393-b374-10c47da55db4'), index=219), arm=BOTH, grasp_description=GraspDescription(approach_direction=<ApproachDirection.FRONT: (<AxisIdentifier.X: (1, 0, 0)>, -1)>, vertical_alignment=<VerticalAlignment.TOP: (<AxisIdentifier.Z: (0, 0, 1)>, -1)>, manipulator=Body(name=PrefixedName('None/milk.stl'), id=UUID('2207bc9f-88b2-4393-b374-10c47da55db4'), index=219), rotate_gripper=False, manipulation_offset=0.05))


In [13]:
pycram_action

PickUpAction(object_designator=Body(name=PrefixedName('None/milk.stl'), id=UUID('2207bc9f-88b2-4393-b374-10c47da55db4'), index=219), arm=BOTH, grasp_description=GraspDescription(approach_direction=<ApproachDirection.FRONT: (<AxisIdentifier.X: (1, 0, 0)>, -1)>, vertical_alignment=<VerticalAlignment.TOP: (<AxisIdentifier.Z: (0, 0, 1)>, -1)>, manipulator=Body(name=PrefixedName('None/milk.stl'), id=UUID('2207bc9f-88b2-4393-b374-10c47da55db4'), index=219), rotate_gripper=False, manipulation_offset=0.05))

## 10. ExecutionLoop — full pipeline with pycram execution

Wire everything together: NL instruction → slot filling → grounding → pycram action → perform.

In [ ]:
from pycram.process_module import simulated_robot

def pycram_executor(preconditions, spec: ActionSpec) -> None:
    """Convert ActionSpec to pycram action and perform it in simulation."""
    pycram_action = bridge.to_pycram_action(spec)
    print(f'[EXECUTOR] Performing {type(pycram_action).__name__}')
    with simulated_robot:
        pycram_action.perform()

loop = ExecutionLoop(
    pipeline=pipeline,
    executor=pycram_executor,
    stop_on_failure=True,
    decomposer=TaskDecomposer(action_type_descriptions=bridge.registry.action_types_dict()),
)
print('ExecutionLoop ready')

In [ ]:
results = loop.run(['pick up the breakfast_cereal from the counter'])

print('\nResults:')
for r in results:
    status = 'OK' if r.success else ('SKIPPED' if r.skipped else f'FAILED: {r.error}')
    print(f'  {r.instruction!r}  →  {status}')

In [ ]:
results = loop.run(['fetch the breakfast_cereal and place it on the island_countertop'])

print('\nResults:')
for r in results:
    status = 'OK' if r.success else ('SKIPPED' if r.skipped else f'FAILED: {r.error}')
    print(f'  {r.instruction!r}  →  {status}')

## 11. EntityGrounder — standalone test

In [ ]:
from krrood.llmr.pipeline.entity_grounder import body_display_name

grounder = EntityGrounder(Body)
result = grounder.ground(EntityDescriptionSchema(role='object_designator', name='milk', semantic_type='Milk'))
print(f'Bodies found: {len(result.bodies)}')
for b in result.bodies:
    print(f'  {body_display_name(b)}')

## 12. SymbolGraph inspection — what krrood sees

In [ ]:
from krrood.symbol_graph.symbol_graph import SymbolGraph
from krrood.llmr.pipeline.entity_grounder import body_display_name, resolve_symbol_class
from krrood.llmr.pipeline.action_pipeline import _is_structural_link

graph = SymbolGraph()
all_bodies = list(graph.get_instances_of_type(Body))
scene_bodies = [b for b in all_bodies if not _is_structural_link(body_display_name(b))]
print(f'Total Body instances : {len(all_bodies)}')
print(f'Scene objects        : {len(scene_bodies)}')
for b in scene_bodies:
    print(f'  {body_display_name(b)}')

In [ ]:
for name in ['DrinkingBottle', 'Cereal', 'CounterTop', 'HasSupportingSurface']:
    cls = resolve_symbol_class(name)
    print(f'  {name!r:30s} → {cls}')

## 13. Debug — Phase 1 + Phase 2 only (no execution)

In [ ]:
instruction = 'pick up the breakfast_cereal from the counter'

schema = pipeline.classify_and_extract(instruction)
print('Phase 1 (slot filling):')
pprint(schema.model_dump())

spec = pipeline.dispatch(schema)
print('\nPhase 2 (dispatch → ActionSpec):')
print(f'  action_type       : {spec.action_type}')
print(f'  parameters        : {spec.parameters}')
print(f'  grounded_entities : {list(spec.grounded_entities.keys())}')

print('\nPhase 3 (ActionSpec → pycram action):')
pycram_action = bridge.to_pycram_action(spec)
print(f'  type  : {type(pycram_action).__name__}')
print(f'  value : {pycram_action}')